# Python `__call__` 学习教程

在 Python 里，很多对象看起来都能“像函数一样被调用”，例如：

- 函数：`f()`
- 类实例：`obj()`
- 类本身：`MyClass()`

它们背后对应的核心机制，就是 `__call__`。

这篇笔记会从三个层次来理解它：

1. 什么是可调用对象
2. 如何给自己的对象实现 `__call__`
3. 元类里的 `__call__` 是怎么控制“创建实例”的


## 1. 什么是可调用对象

所谓“可调用”，就是这个对象后面可以直接加一对括号：`obj(...)`。

Python 提供了一个内置函数 `callable()`，可以判断对象是否支持这种调用方式。

常见的可调用对象包括：

- 函数
- 类
- 实现了 `__call__` 的实例
- 一些内置函数和方法


In [1]:
def add(a, b):
    return a + b

class Greeter:
    pass

g = Greeter()

print(callable(add))
print(callable(Greeter))
print(callable(g))
print(callable(len))


True
True
False
True


运行结果通常是：

- `add` 是可调用的，因为它是函数
- `Greeter` 是可调用的，因为类本身可以用来创建实例
- `g` 默认不可调用，因为它只是一个普通实例
- `len` 也是可调用的，因为它本质上是一个内置函数

这里要记住一个很重要的点：

`callable(obj)` 只是在问“它能不能被 `()` 调用”，并不是在问“它是不是函数”。


## 2. 给对象实现 `__call__`

如果一个类的实例想要像函数一样使用，就可以在类里定义 `__call__` 方法。

这样一来：

- `obj = MyClass()` 创建对象
- `obj()` 会触发 `obj.__call__()`

这让对象既能保存状态，又能像函数一样执行逻辑。


In [7]:
class Adder:
    def __init__(self, base):
        self.base = base

    def __call__(self, value):
        return self.base + value

add10 = Adder(10)

print(callable(add10))
print(add10(5))
print(add10(100))
print(add10.__call__(100))


True
15
110
110


这个例子里，`Adder(10)` 返回的是一个对象，但这个对象又能像函数一样被调用。

它的好处是：

- 能保存内部状态，比如这里的 `base`
- 调用接口很自然，像普通函数一样
- 很适合做带配置的函数对象、回调对象、策略对象


In [5]:
class Counter:
    def __init__(self):
        self.count = 0

    def __call__(self):
        self.count += 1
        return self.count

counter = Counter()

print(counter())
print(counter())
print(counter())


1
2
3


这个例子说明 `__call__` 还可以保存“调用次数”这样的状态。

所以，`__call__` 不只是“让对象能被调用”，更重要的是：**它让对象有机会在每次调用时执行自定义逻辑**。


## 3. `__call__` 与函数对象

从使用体验上看，函数本身也是一种可调用对象。

也就是说，你平时写的：

```python
def f(x):
    return x + 1
```

本质上也是一个“能被调用的对象”。

你可以把它理解成：

- 普通函数：可调用对象的一种
- 自定义类实例：也可以通过 `__call__` 变成可调用对象

两者在“调用”这一点上是统一的。


In [6]:
def hello(name):
    return f"Hello, {name}!"

print(hello("Python"))
print(type(hello))
print(callable(hello))
print(hello.__call__("World"))


Hello, Python!
<class 'function'>
True
Hello, World!


这里可以看到，函数对象自己也有 `__call__`。

所以从概念上说，`()` 并不是“函数专属语法”，而是 Python 的“调用协议”。只要对象实现了这个协议，它就能被调用。


## 4. `__call__` 的常见用途

### 4.1 带状态的函数

有些逻辑需要在多次调用之间保留状态，比如计数、缓存、配置参数等。用普通函数也能做，但用可调用对象往往更清晰。

### 4.2 回调对象

在事件系统、任务调度器、数据处理管道里，经常需要传入一个“稍后会被执行的对象”。这种场景里，可调用对象很自然。

### 4.3 简化复杂接口

如果一个对象背后有很多准备工作，但真正使用时只想写成 `obj()`，那么 `__call__` 能把“初始化”和“执行”分开，让接口更简洁。


## 5. 类对象的 `__call__`：元类层面

前面讲的是“实例的 `__call__`”。

但类本身也能被调用：`MyClass()`。
这件事不是普通实例的 `__call__` 负责的，而是由类的元类来控制。

默认情况下，类的元类是 `type`。当你写：

```python
obj = MyClass()
```

底层其实会走元类的 `__call__`，然后再间接触发：

1. `__new__` 创建实例
2. `__init__` 初始化实例

所以，元类的 `__call__` 可以看作“实例化入口”。


In [11]:
class Demo:
    def __new__(cls, *args, **kwargs):
        print("__new__ called")
        return super().__new__(cls)

    def __init__(self, value):
        print("__init__ called")
        self.value = value

d = Demo(123)
print(d.value)


__new__ called
__init__ called
123


这段代码有助于理解“创建对象”这件事的执行顺序。

你可以把它记成：

- `Demo(...)` 先触发类的调用
- 类的调用交给元类处理
- 元类再负责真正创建实例
- 创建好之后再调用 `__init__`

所以，类的 `()` 和实例的 `()` 不是一回事，只是表面语法相同。


## 6. 用元类重写 `__call__`

元类里重写 `__call__`，可以在对象实例化时插入自己的逻辑。

最常见的例子之一是单例模式：不管调用多少次类，永远返回同一个实例。


In [ ]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]


class Database(metaclass=SingletonMeta):
    def __init__(self):
        print("Database initialized")


db1 = Database()
db2 = Database()

print(db1 is db2)


这个例子里，`Database()` 被调用了两次，但真正初始化只发生了一次。

原因是元类 `SingletonMeta.__call__` 把“是否创建新实例”的决定权接管了。

这也是 `__call__` 在高级 Python 编程里很有价值的地方：它可以拦截对象创建过程。


## 7. 小结

你可以把 `__call__` 记成一句话：

**让对象支持括号调用。**

进一步拆开看：

- 普通实例的 `__call__`：让对象像函数一样工作
- 函数本身：天然就是可调用对象
- 元类的 `__call__`：控制类的实例化过程

如果你只记住一件事，那就是：

`()` 不是函数专属，`__call__` 才是 Python 调用机制的核心接口。
